# Experiment 3: Regression Analysis using Linear and Regularized Models
**ICS1512 – Machine Learning Algorithms Laboratory**

Dataset: Loan Amount prediction (Kaggle "Predict Loan Amount Data")

In [ ]:
import numpy as np
import pandas as pd
import time
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_style("whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 1. Load Dataset

The Kaggle "Predict Loan Amount Data" zip contains `train.csv` and `test.csv`. Only `train.csv`
has the target column (`Loan Sanction Amount (USD)`) filled in -- `test.csv` is an unlabeled
Kaggle-submission file with no target, so it cannot be used to compute regression metrics. This
notebook therefore uses **`train.csv` only**, and creates its own train/test split from it (as the
lab manual requires for evaluation).

Place `train.csv` in the same folder as this notebook.

In [ ]:
DATA_PATH = "train.csv"

df = pd.read_csv(DATA_PATH)
target = "Loan Sanction Amount (USD)"

# Identifier / free-text columns that carry no predictive signal
id_cols = ["Customer ID", "Name", "Property ID"]
df = df.drop(columns=[c for c in id_cols if c in df.columns])

print(df.shape)
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print(df.shape)
df.info()
df.describe(include="all")

In [ ]:
print("Missing values per column:")
print(df.isnull().sum())

### Target variable distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.histplot(df[target].dropna(), kde=True, color="steelblue")
plt.title("Distribution of Loan Sanction Amount (Target)")
plt.xlabel(target)
plt.tight_layout()
plt.show()

### Feature vs. target scatter plots

In [ ]:
num_features = [c for c in [
    "Income (USD)", "Loan Amount Request (USD)", "Credit Score", "Property Price"
] if c in df.columns]

fig, axes = plt.subplots(1, len(num_features), figsize=(5 * len(num_features), 4))
if len(num_features) == 1:
    axes = [axes]
for ax, feat in zip(axes, num_features):
    ax.scatter(df[feat], df[target], alpha=0.4, color="teal", s=10)
    ax.set_xlabel(feat)
    ax.set_ylabel(target)
    ax.set_title(f"{feat} vs Target")
plt.tight_layout()
plt.show()

## 3. Data Preprocessing
- Handle missing values (median for numeric, mode for categorical)
- Encode categorical variables (one-hot encoding)
- Standardize numerical features

In [ ]:
df = df.dropna(subset=[target]).reset_index(drop=True)

cat_cols = df.select_dtypes(include="object").columns.tolist()
num_cols = [c for c in df.select_dtypes(include=np.number).columns if c != target]

for c in num_cols:
    df[c] = df[c].fillna(df[c].median())
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])

df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

X = df_encoded.drop(columns=[target])
y = df_encoded[target]

print("Feature matrix shape:", X.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

scaler = StandardScaler()
num_cols_present = [c for c in num_cols if c in X_train.columns]

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()
X_train_scaled[num_cols_present] = scaler.fit_transform(X_train[num_cols_present])
X_test_scaled[num_cols_present] = scaler.transform(X_test[num_cols_present])

print("Train shape:", X_train_scaled.shape, "| Test shape:", X_test_scaled.shape)

## 4. Baseline Linear Regression

In [ ]:
start = time.time()
lin_reg = LinearRegression()
lin_reg.fit(X_train_scaled, y_train)
lin_train_time = time.time() - start

y_pred_lin = lin_reg.predict(X_test_scaled)
print(f"Training time: {lin_train_time:.4f}s")

## 5. Hyperparameter Tuning (Ridge, Lasso, Elastic Net)
5-Fold Cross-Validation Grid Search over the specified hyperparameter grids.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

param_grids = {
    "Ridge": {"alpha": [0.01, 0.1, 1, 10, 100]},
    "Lasso": {"alpha": [0.001, 0.01, 0.1, 1, 10]},
    "ElasticNet": {"alpha": [0.01, 0.1, 1, 10], "l1_ratio": [0.2, 0.5, 0.8]},
}

estimators = {
    "Ridge": Ridge(random_state=RANDOM_STATE),
    "Lasso": Lasso(random_state=RANDOM_STATE, max_iter=10000),
    "ElasticNet": ElasticNet(random_state=RANDOM_STATE, max_iter=10000),
}

best_models = {}
tuning_summary = []

for name, estimator in estimators.items():
    grid = GridSearchCV(
        estimator, param_grids[name], scoring="r2", cv=cv, n_jobs=-1
    )
    start = time.time()
    grid.fit(X_train_scaled, y_train)
    fit_time = time.time() - start

    best_models[name] = grid.best_estimator_
    tuning_summary.append({
        "Model": name,
        "Search Method": "Grid Search",
        "Best Parameters": grid.best_params_,
        "Best CV R2": grid.best_score_,
        "Training Time (s)": fit_time,
    })

tuning_df = pd.DataFrame(tuning_summary)
tuning_df

## 6. Train Final Models

In [ ]:
models = {
    "Linear Regression": lin_reg,
    "Ridge Regression": best_models["Ridge"],
    "Lasso Regression": best_models["Lasso"],
    "Elastic Net Regression": best_models["ElasticNet"],
}

train_times = {"Linear Regression": lin_train_time}
for name in ["Ridge Regression", "Lasso Regression", "Elastic Net Regression"]:
    key = name.split()[0] if name != "Elastic Net Regression" else "ElasticNet"
    row = tuning_df.loc[tuning_df["Model"] == key, "Training Time (s)"]
    train_times[name] = float(row.values[0]) if len(row) else np.nan

predictions = {name: model.predict(X_test_scaled) for name, model in models.items()}

## 7. Cross-Validation Performance (K = 5)

In [ ]:
def cv_metrics(model, X, y, cv):
    mae = -cross_val_score(model, X, y, scoring="neg_mean_absolute_error", cv=cv).mean()
    mse = -cross_val_score(model, X, y, scoring="neg_mean_squared_error", cv=cv).mean()
    rmse = np.sqrt(mse)
    r2 = cross_val_score(model, X, y, scoring="r2", cv=cv).mean()
    return mae, mse, rmse, r2

cv_results = []
for name, model in models.items():
    mae, mse, rmse, r2 = cv_metrics(model, X_train_scaled, y_train, cv)
    cv_results.append({"Model": name, "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2})

cv_results_df = pd.DataFrame(cv_results)
cv_results_df

## 8. Test Set Performance

In [ ]:
test_results = []
for name, model in models.items():
    y_pred = predictions[name]
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    test_results.append({
        "Model": name, "MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2,
        "Training Time (s)": train_times[name],
    })

test_results_df = pd.DataFrame(test_results)
test_results_df

### Test Set Performance Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(test_results_df["Model"], test_results_df["RMSE"], color="indianred")
axes[0].set_title("RMSE by Model")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(test_results_df["Model"], test_results_df["R2"], color="seagreen")
axes[1].set_title("R2 Score by Model")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## 9. Predicted vs. Actual and Residual Plots

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 9))

for i, (name, y_pred) in enumerate(predictions.items()):
    # Predicted vs actual
    ax = axes[0, i]
    ax.scatter(y_test, y_pred, alpha=0.5, color="steelblue")
    lims = [min(y_test.min(), y_pred.min()), max(y_test.max(), y_pred.max())]
    ax.plot(lims, lims, "r--", linewidth=1)
    ax.set_title(f"{name}\nPredicted vs Actual")
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")

    # Residuals
    residuals = y_test - y_pred
    ax2 = axes[1, i]
    ax2.scatter(y_pred, residuals, alpha=0.5, color="darkorange")
    ax2.axhline(0, color="red", linestyle="--", linewidth=1)
    ax2.set_title(f"{name}\nResidual Plot")
    ax2.set_xlabel("Predicted")
    ax2.set_ylabel("Residual")

plt.tight_layout()
plt.show()

## 10. Training Error vs. Validation Error (Overfitting/Underfitting)

In [ ]:
from sklearn.model_selection import learning_curve

fig, axes = plt.subplots(1, 4, figsize=(22, 4.5))

for ax, (name, model) in zip(axes, models.items()):
    train_sizes, train_scores, val_scores = learning_curve(
        model, X_train_scaled, y_train, cv=cv,
        scoring="neg_root_mean_squared_error",
        train_sizes=np.linspace(0.1, 1.0, 6), random_state=RANDOM_STATE
    )
    train_err = -train_scores.mean(axis=1)
    val_err = -val_scores.mean(axis=1)

    ax.plot(train_sizes, train_err, "o-", label="Training error")
    ax.plot(train_sizes, val_err, "o-", label="Validation error")
    ax.set_title(name)
    ax.set_xlabel("Training set size")
    ax.set_ylabel("RMSE")
    ax.legend()

plt.tight_layout()
plt.show()

## 11. Effect of Regularization on Coefficients

In [ ]:
coef_df = pd.DataFrame({
    "Feature": X_train_scaled.columns,
    "Linear": models["Linear Regression"].coef_,
    "Ridge": models["Ridge Regression"].coef_,
    "Lasso": models["Lasso Regression"].coef_,
    "Elastic Net": models["Elastic Net Regression"].coef_,
})

top_features = coef_df.reindex(
    coef_df["Linear"].abs().sort_values(ascending=False).index
).head(10)

top_features.set_index("Feature").plot(kind="bar", figsize=(14, 5))
plt.title("Coefficient Comparison Across Models (Top 10 Features by |Linear coef|)")
plt.ylabel("Coefficient value")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

top_features

In [ ]:
n_nonzero = {
    "Linear": np.sum(models["Linear Regression"].coef_ != 0),
    "Ridge": np.sum(models["Ridge Regression"].coef_ != 0),
    "Lasso": np.sum(models["Lasso Regression"].coef_ != 0),
    "Elastic Net": np.sum(models["Elastic Net Regression"].coef_ != 0),
}
print("Number of non-zero coefficients per model:")
n_nonzero

## 12. Overfitting and Underfitting Analysis

- **Training vs. validation error:** compare the gap between the training and validation error
  curves above. A large gap (low training error, high validation error) indicates overfitting;
  high error on both indicates underfitting.
- **Effect of regularization strength:** larger `alpha` shrinks coefficients more aggressively,
  increasing bias but reducing variance. Small `alpha` behaves close to plain Linear Regression.
- **Improvement after tuning:** compare the CV R2 in the tuning summary table to the test R2 in the
  performance table — tuned Ridge/Lasso/Elastic Net should generalize better than an untuned model
  with an extreme alpha.

## 13. Bias–Variance Analysis

- **Linear Regression:** typically low bias, higher variance — fits training data closely but can
  be sensitive to noise and multicollinearity, hurting generalization.
- **Ridge / Elastic Net:** shrink coefficients (L2 penalty), trading a small increase in bias for a
  meaningful reduction in variance, usually improving test performance.
- **Lasso (and the L1 part of Elastic Net):** drives some coefficients exactly to zero, performing
  feature selection and increasing model sparsity, which can further reduce variance at the cost of
  bias if useful features are zeroed out.

## 14. Observations and Conclusion

Summarize, using the tables and plots above:
- Which model achieved the best test R2 / RMSE.
- The optimal hyperparameters found via Grid Search (Table 1 equivalent, `tuning_df`).
- Whether regularization improved generalization compared to plain Linear Regression.
- The trade-off observed between model accuracy and complexity (number of non-zero coefficients,
  training time).

In [ ]:
summary = test_results_df.sort_values("R2", ascending=False).reset_index(drop=True)
print("Final ranking by test R2:")
summary